# 15 · 第三阶段开工前的前提查验

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → 第二阶段 F ✅（14a–14f + 附录 14x）从一个分数到两个能量
                 ↓
      第三阶段 G：从"给已有的序列打分"到"自己提出新序列"
      └─ 15 前提查验  ← 你在这里（动手之前的闸门）
```

## 第三阶段想做什么

前两个阶段做的是**打分器**：给一条蛋白序列，输出这个突变有多坏。
第三阶段想反过来——**让模型自己提出序列**。这就是"生成式蛋白设计"。

但这个想法压着两个假设。**假设没查就是空想**，所以先查，查不过就改方案。

| 假设 | 查不过会怎样 |
|---|---|
| 湖里存着大量"两个能量"的打分，可以拿来训练生成器 | 如果那些蛋白是考试蛋白，等于把考卷喂给模型，后面所有验证作废 |
| 多突变的实验数据够训一个多突变打分器 | 生成器一次会改很多个位置，只在单突变上验证过的打分器不能信 |

**这一章的结论是负面的。** 负面结果照写——08b、08c 都是这么写的。

---

# 背景

这一章出现的每个概念，在这里讲一遍。都是陈述，不需要任何生物学基础。
**看完这一节，后面每一步的数字你都能自己判断对不对。**

## 1 · 蛋白质、序列、位置

蛋白质是一条链，由 **20 种氨基酸**串成，每种用一个大写字母表示（A、C、D、E、F…）。
一条蛋白就是一串字母，比如 `MKTAYIAKQRQISFVK...`，长度从几十到几千。

每个氨基酸占一个**位置**，从 1 开始编号。"第 73 位是 K"就是说这串字母的第 73 个是 K。

这条链会自己卷成一个**固定的三维形状**，这个过程叫**折叠**。
**只有折叠成对的形状，蛋白才能干活。** 折不起来的蛋白，细胞会把它当垃圾降解掉。

## 2 · 突变的写法

**突变**就是把某个位置的字母换成另一个。

- `K73A` = 第 73 位原本是 K，换成 A。这是**单突变**。
- `K73A:S130G` = 同时换两个位置。这是**双突变**，冒号分隔。

本章会一直用这个记号。

## 3 · 稳定性，和 ΔΔG 这个数

蛋白在细胞里不是永远折叠着的，它在"折叠"和"散开"两个状态之间来回。
**稳定性**就是指它有多倾向于保持折叠。

- 稳定的蛋白：绝大部分时间折叠着。
- 不稳定的蛋白：经常散开，于是经常被降解掉。

**ΔΔG 衡量一个突变让稳定性变了多少**，单位是 **kcal/mol**（一个能量单位）。

本章数据集的符号约定（不同论文符号可能相反，这里按原数据）：

| ΔΔG | 含义 |
|---|---|
| **正数** | 突变让蛋白**更稳定** |
| **负数** | 突变让蛋白**更不稳定** |
| 0 附近 | 基本没影响 |

典型范围大约 **−5 到 +3**。这个范围的上下限待会儿很重要。

## 4 · 这些数是怎么测出来的（这一节是全章的关键）

**不讲这个，后面"量程"那部分就只能靠我断言。**

Tsuboyama 等人 2023 年用的方法，原理是这样：

1. 造出几十万个突变体，每个蛋白一条链。
2. 加入**蛋白酶**——一种专门剪断蛋白的酶。
3. **关键**：折叠着的蛋白剪不动（结构把链保护起来了）；散开的蛋白一剪就断。
4. 所以：**蛋白越不稳定 → 散开的时间越多 → 被剪断得越多 → 存活下来的越少。**
5. 测每个突变体的存活比例，再反推出它的 ΔΔG。

**为什么会有量程**：

这个读数是"存活比例"，范围只能是 0 到 1。
一个蛋白如果已经**完全不折叠**了，它的存活比例就是 0。
这时候你再让它更不稳定一点——存活比例还是 0，**读数不会再动**。

于是：**超过某个程度之后，实验就分不出"很差"和"非常差"了。**
反推出来的 ΔΔG 会卡在一个底部数值附近，不再是真实的能量。

这就是本章第 4 步要查的东西。记住这个机制，到时候你自己就能判断证据站不站得住。

## 5 · 可加性，和上位效应

如果 `K73A` 单独测是 **−1.0**，`S130G` 单独测是 **−1.5**，
那么双突变 `K73A:S130G` 会不会正好是 **−2.5**？

- 是 → 叫**可加**（两个突变各管各的，互不干扰）。
- 不是 → 差出来的那部分叫**上位效应**（epistasis），意思是两个突变**互相影响**。

**为什么关心这个**：真的上位效应通常需要两个位置在三维空间里**挨得近**——
挨着才能互相影响。这条性质在第 4 步会被用来做判据。

## 6 · 训练集、考试集、泄漏

机器学习的基本规矩：

- **训练集**（train split）：模型可以学的数据。
- **考试集**（test split）：模型不许见到，用来打分。
- **泄漏**（leakage）：考试集的信息混进了训练，分数就虚高，不作数。

这个项目在 07a 把考试蛋白**锁死**了，而且规定**考卷只开一次**。
不只按蛋白切，还按**同源**切——序列相似的蛋白必须在同一边，
否则模型靠"见过它的近亲"就能蒙对。

**这一章第 1 步查的就是：第三阶段会不会破坏这条规则。**

## 7 · 第三阶段想加的东西：生成式蛋白设计

前两阶段做的是**打分器**：

```
输入：一条序列 + 一个突变   →   输出：这个突变有多坏
```

**生成式**是反过来：

```
输入：一个三维形状（主链）   →   输出：能折成这个形状的序列
```

这类模型叫**反向折叠**（inverse folding），代表作是 ProteinMPNN。
"反向"的意思是：正常是"序列决定形状"，它反着来，从形状推序列。

**为什么这对我们有意义**：现有的反向折叠模型只管"折得起来"，
不管"折起来之后还能不能干活"。而第二阶段的产出——**两个能量**
（一个管折叠、一个管功能）——正好补这一块。

## 8 · "两个能量"是什么

第二阶段（14c）的模型对每个突变输出**两个**数，不是一个：

| | 含义 | 用什么实验数据监督 |
|---|---|---|
| **折叠能** ΔΔG_fold | 这个突变让蛋白**折叠**变难多少 | 测蛋白量的实验 |
| **功能能** ΔΔG_func | 折叠好了之后，**干活**的能力损失多少 | 测活性的实验 |

两个都是 kcal/mol。这个拆分不是网络结构暗示的，是**由实验类型强制的**。

这一章第 1 步要查的，就是这批"两个能量"的打分覆盖了哪些蛋白。

---

## 这一章做什么

| 步骤 | 内容 |
|---|---|
| 1 | 查**湖里的两个能量覆盖了哪些蛋白**——是不是考试蛋白 |
| 2 | 查**多突变数据有多少**——够不够训一个模型 |
| 3 | 查**头部空间**——不用模型、直接把两个单突变的实测值相加，够不够好 |
| 4 | 查**差距的来源**——是真的上位效应，还是实验量程的问题 |
| 5 | 查**稳定端的数据量**——够不够做回溯验证 |

In [ ]:
import os
import pickle
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
WORK = ROOT / "data/stability"   # 14a 的本机工作副本，不进 Git

CSV = WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv"
SPLITS = WORK / "mega_splits.pkl"
SINGLE = re.compile(r"^([A-Z])(\d+)([A-Z])$")
print(CSV.exists(), SPLITS.exists())

---

## 第 1 步 · 湖里的两个能量，覆盖了哪些蛋白

### 在干什么
确认一件事：数据湖里那批"两个能量"的打分，是给哪些蛋白打的。

### 为什么要查
第三阶段的初稿想拿这批打分当训练数据，去训一个"以能量为条件"的生成器。
**但训练数据里不能有考试蛋白。**

### 怎么查
不用跑代码。14d 的记账和 README 已经写明，把两个数字放在一起看就够了。

| | 初稿以为的 | 实际 |
|---|---|---|
| 湖里那 2.17 亿条打分 | 两个能量 | **09c 的旧模型，只有一个分数** |
| 两个能量重打的范围 | 接近全蛋白组 | **118 个蛋白 / 103,744 个位置 / 197 万个替换**（14d） |
| 那 118 个蛋白是谁 | 任意一批 | **108 个是锁死的考试蛋白** + KRAS、ABL1、SHP2 |

### 结果怎么看
- **湖里的两个能量，基本上就是考试集本身。**
- 拿它训练任何下游模型，等于把考卷提前给模型看。
- 之后在考试蛋白上做的回溯验证，全部作废。

### 这一步的结论
初稿那条路**堵死**，不是打补丁的问题，是换底座。
替代底座只能是 **train split 蛋白的实测值**——于是要先知道实测的多突变有多少，就有了第 2 步。

### 面试怎么说
> "Our own model's outputs cover the locked test proteins, so they cannot train a downstream
> generator; the replacement supervision has to be measured values on training-split proteins."

---

## 第 2 步 · 多突变数据有多少

### 在干什么
数一数 Tsuboyama 数据里有多少**双突变及以上**的测量值。

### 为什么要查
生成器一次会改很多个位置。而第二阶段的打分器只在**单突变**上训练和验证过
（14c 虽然做过"单突变相加预测双突变"，但那是在一个训练蛋白上检验可加性本身，不是训练多突变模型）。
**只在单突变上验证过的打分器，不能用来评判多突变的设计。**

项目的 `stability.megascale` 在复现 ThermoMPNN 的数据清洗流程时，有一步**同时**丢掉了
插入、缺失和多突变（607,839 → 391,090）。那一步丢掉的，正是这里要用的料。

### 怎么保证数得对
**先自检清洗流程**：三个已发表的数字必须一个不差地复现出来。
对不上就说明数据或代码变了，后面的计数都不可信。

### 执行
读原始 CSV，按 ThermoMPNN 的三步过滤，核对每一步的行数。

In [ ]:
# 读一次，后面各步复用（697 MB，约半分钟）
raw = pd.read_csv(CSV, usecols=["ddG_ML", "mut_type", "WT_name", "aa_seq"], low_memory=False)

print(f"{'文件行数':<24}{len(raw):>10,}   已发表 776,298")
withddg = raw[raw.ddG_ML != "-"]
print(f"{'有可靠 ddG':<22}{len(withddg):>10,}   已发表 607,839")

indel = withddg.mut_type.str.contains("ins|del", regex=True)
multi = ~indel & withddg.mut_type.str.contains(":")
single = ~indel & ~multi & (withddg.mut_type != "wt")
wt = withddg.mut_type == "wt"

print(f"{'  插入 / 缺失':<21}{int(indel.sum()):>10,}")
print(f"{'  多突变':<23}{int(multi.sum()):>10,}")
print(f"{'  单突变':<23}{int(single.sum()):>10,}")
print(f"{'  野生型':<23}{int(wt.sum()):>10,}")
print(f"{'  单突变 + 野生型':<19}{int(single.sum() + wt.sum()):>10,}   已发表 391,090")

### 结果怎么看
三个数字全部对上，计数可信。

### 执行（续）
把多突变按"所在结构域属于哪个 split"分组。

In [ ]:
class _NoCodeUnpickler(pickle.Unpickler):
    """只允许 numpy 的数组机制：pickle 是程序不是数据文件，不给它执行任何东西的机会。"""

    ALLOWED = {
        ("numpy.core.multiarray", "_reconstruct"),
        ("numpy._core.multiarray", "_reconstruct"),
        ("numpy", "ndarray"),
        ("numpy", "dtype"),
    }

    def find_class(self, module, name):
        if (module, name) not in self.ALLOWED:
            raise pickle.UnpicklingError(f"refused to import {module}.{name}")
        return getattr(__import__(module, fromlist=[name]), name)


with open(SPLITS, "rb") as f:
    splits = {k: [str(x) for x in v] for k, v in _NoCodeUnpickler(f).load().items() if k in ("train", "val", "test")}
where = {name: split for split, names in splits.items() for name in names}

m = withddg[multi].copy()
m["k"] = m.mut_type.str.count(":") + 1
m["split"] = m.WT_name.map(where)

print("按突变个数 k:", m.k.value_counts().sort_index().to_dict())
print()
summary = m.groupby(m.split.fillna("不在已发表 split 内")).agg(行数=("mut_type", "size"), 结构域=("WT_name", "nunique"))
summary.loc["可用于训练 (train+val)"] = [
    int(m.split.isin(["train", "val"]).sum()),
    m[m.split.isin(["train", "val"])].WT_name.nunique(),
]
print(summary.to_string())
print()
for name, names in splits.items():
    print(f"  {name:<6}{m[m.WT_name.isin(names)].WT_name.nunique():>4} / {len(set(names)):>4} 个结构域有多突变")

### 结果怎么看
- **159,051 行多突变，分布在 153 个结构域上。**
- 可用于训练 **136,970 行 / 133 个结构域**；留出 **22,081 行 / 20 个结构域**当考卷。
- 按结构域切分和 14a 完全一致，现有的防泄漏规则直接套用。

### 关键：全部是双突变
`k` 的分布只有一个值：**2**。一个三突变及以上都没有。

这意味着这批数据能验证"**单突变 → 双突变**"，**验证不了**"双突变 → 十几二十个突变"。
设计产出要么限制在少量突变，要么必须标注成外推、没有实测支撑。**这条边界不许含糊。**

还要再砍一刀：14a 查出 56 个结构域和锁死的考试蛋白同源，其中 **52 个在 Tsuboyama 的训练侧**。
这 133 个可用结构域要用同一条同源规则再过一遍，预计掉 15–20%。

### 面试怎么说
> "The multi-mutant pool the standard filter chain discards is 159,051 rows on 153 domains,
> but every one of them is a double, so anything beyond two substitutions is extrapolation."

---

## 第 3 步 · 头部空间有多大

### 在干什么
比较两件事：
- **免费方案**：把两个组分单突变的**实测**能量直接相加，当作双突变的预测值。
- **实测的双突变能量**。

### 为什么要查
"数据够多"不等于"值得做一章"。要做一个多突变模型，前提是**免费方案不够用**。
如果直接相加就已经很准，那这个模型没有存在价值。

这是项目一贯的规矩：**必须先赢最简单的基线**（14a 先赢"保守性 × 埋藏程度"，也是这个道理）。

### 为什么这一步不会泄漏
全程只用**实测值**，不碰任何模型。没有可泄漏的东西。

### 先清理数据
双突变写成 `T9Q:N23Q`。注意有 `T9Q:N23N` 这种——第二个改动是 N 换成 N，**没有变**，
那其实是单突变，必须滤掉。

### 执行
拆开双突变、滤掉同义的、把两个组分的实测单突变值接上来。

In [ ]:
d = m.copy()
d["ddg"] = pd.to_numeric(d.ddG_ML)
parts = d.mut_type.str.split(":", expand=True)
for i in (0, 1):
    p = parts[i].str.extract(SINGLE)
    d[[f"wt{i}", f"pos{i}", f"mut{i}"]] = p
    d[f"pos{i}"] = pd.to_numeric(d[f"pos{i}"])
d = d.dropna(subset=["wt0", "wt1"])

synonymous = (d.wt0 == d.mut0) | (d.wt1 == d.mut1)
print(f"池子里的双突变          {len(d):>9,}")
print(f"  有同义组分 (X->X)   {int(synonymous.sum()):>9,}   <- 其实是单突变，滤掉")
d = d[~synonymous].copy()
print(f"  真正的双替换        {len(d):>9,}")

# 单突变的实测值，用来相加
s = withddg[single].copy()
s = s[s.mut_type.str.match(SINGLE)]
singles = s.assign(ddg_single=pd.to_numeric(s.ddG_ML)).set_index(["WT_name", "mut_type"]).ddg_single

for i in (0, 1):
    name = d[f"wt{i}"] + d[f"pos{i}"].astype(str) + d[f"mut{i}"]
    d[f"ddg{i}"] = singles.reindex(pd.MultiIndex.from_arrays([d.WT_name, name])).values

both = d.dropna(subset=["ddg0", "ddg1"]).copy()
both["additive"] = both.ddg0 + both.ddg1
both["residual"] = both.ddg - both.additive
both["separation"] = (both.pos1 - both.pos0).abs()
print(f"两个组分都有实测值的   {len(both):>9,}   ({len(both) / len(d):.1%})，{both.WT_name.nunique()} 个结构域")

### 执行（续）
算相关性、R²、RMSE 和残差（实测 − 相加）。

In [ ]:
def report(frame, label):
    x, y = frame.additive.to_numpy(), frame.ddg.to_numpy()
    ss_res = float(((y - x) ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return {
        "窗口": label,
        "n": len(frame),
        "结构域": frame.WT_name.nunique(),
        "Pearson": round(stats.pearsonr(x, y)[0], 3),
        "Spearman": round(stats.spearmanr(x, y)[0], 3),
        "R2": round(1 - ss_res / ss_tot, 3),
        "RMSE": round(np.sqrt(ss_res / len(frame)), 3),
        "偏差": round(frame.residual.mean(), 3),
        "残差SD": round(frame.residual.std(), 3),
    }


rows = [report(both, "全部")] + [
    report(both[both.split == n], f"已发表 {n} split") for n in ("train", "val", "test")
]
print(pd.DataFrame(rows).to_string(index=False))

### 结果怎么看：一个矛盾，必须解释

| 指标 | 值 | 含义 |
|---|---|---|
| Pearson / Spearman | **0.81** | 排序很好——谁比谁更不稳定，基本排对了 |
| **R²** | **−0.42** | 比"直接猜平均值"还差 |
| 残差均值 | **+0.894 kcal/mol** | 这就是原因 |

R² 是负的，说明预测值系统性地偏离实测值。而残差均值 +0.894 说明**不是散乱的误差，是整体偏移**：
相加出来的值，系统性地比实测值更低（更不稳定）。

### 两个候选解释
| 解释 | 如果是它，该有什么特征 |
|---|---|
| **真的上位效应**：两个突变互相影响 | 残差应该在**两个位置靠得近**的时候更大（要接触才能互相影响） |
| **测量被压顶**：超出量程 | 残差应该随**预测的严重程度**单调增长，而和位置间距无关 |

区分这两个，决定第三阶段能做什么。所以有了第 4 步。

### 面试怎么说
> "Parameter-free additivity ranks doubles at 0.81 Spearman but scores R-squared −0.42;
> the gap is a systematic +0.89 kcal/mol bias, not scatter."

---

## 第 4 步 · 那个差距是上位效应，还是测量问题

### 在干什么
用两个诊断把上面两个解释分开。

### 怎么分
1. **残差 vs 两个位置的间距**：真的上位效应需要两个位置在空间上接触。
   相距很远还一样大，就不是上位效应。
2. **残差 vs 预测的严重程度**：量程压顶的特征是——预测越惨烈，被压得越多。

### 执行

In [ ]:
print("=== 残差 vs 位置间距（真上位效应的签名）===")
bins = pd.cut(both.separation, [0, 4, 8, 16, 32, 10_000], labels=["1-4", "5-8", "9-16", "17-32", ">32"])
print(both.groupby(bins, observed=True).residual.agg(["size", "mean", "std"]).round(3).to_string())

print()
print("=== 残差 vs 可加预测值（压顶的签名）===")
print(f"相关系数 {stats.pearsonr(both.additive, both.residual)[0]:+.3f}")
print()
print(
    both.groupby(pd.qcut(both.additive, 8), observed=True)
    .agg(n=("ddg", "size"), 可加预测=("additive", "mean"), 实测=("ddg", "mean"), 残差=("residual", "mean"))
    .round(3)
    .to_string()
)

### 结果怎么看

**间距那一侧（上位效应的特征）：不成立。**
- 相距 1–4 个残基：残差 0.670
- 相距 **>32** 个残基：残差 **0.980**

相距 32 个残基以上基本不会接触，真的上位效应应该消失。**它没有，反而更大。**

**严重程度那一侧（压顶的特征）：成立，而且很强。**
- 残差和相加预测值的相关系数 **−0.72**
- 预测 −0.55 时，残差 +0.11
- 预测 −5.55 时，残差 **+2.23**

单调增长，毫无含糊。

### 执行（续）
再确认一件事：实测值有没有一个硬地板。

In [ ]:
print("=== 实测 ddG 有没有硬地板 ===")
for q in (0, 0.1, 0.5, 1, 2, 5):
    print(f"  第 {q:>3}% 分位   {both.ddg.quantile(q / 100):>7.3f}")
print(f"  双突变最低 {both.ddg.min():.2f}   最高 {both.ddg.max():.2f}")
print(f"  单突变最低 {both.ddg0.min():.2f}   最高 {both.ddg0.max():.2f}   <- 同一个地板")

deep = both[both.additive < -5]
mid = both[(both.additive > -1) & (both.additive < 1)]
print()
print(f"  可加预测 < -5 的 {len(deep):,} 行：预测均值 {deep.additive.mean():.2f}，实测均值 {deep.ddg.mean():.2f}，残差 {deep.residual.mean():+.2f}")
print(f"  对照（可加预测 -1~1）：实测均值 {mid.ddg.mean():.2f}，残差 {mid.residual.mean():+.2f}")

### 结果怎么看
- 双突变实测的最低值 **−5.69**
- 单突变实测的最低值 **−5.52**
- **同一个地板。** 这是实验的量程，不是蛋白的性质。
- 相加预测 −5.98 的那 10,696 行，实测平均只有 **−3.52**。

### 关键结论
**那 0.9 kcal/mol 的"非可加性"，绝大部分是测量截断，不是生物学。**

两个失稳突变相加经常预测到 −6 到 −10，超出这个实验能测的范围，实测就饱和在 −5 附近。

### 一个重要的补充：这不是新发现
Otwinowski 2018（*Mol Biol Evol*）在 GB1 上已经报告过同一个机制、同一个方向：
测量地板"produces large amounts of non-biological positive pairwise epistasis"。
他的处理是把地板写进似然，并把靠近地板的配对**剔除**。

**我们做的是量化和推广**：他从未把这件事命名、没有推广到整个数据集，
也没有做"残差 vs 间距"这个分离检验。

### 执行（续）
既然知道是量程问题，那把超出量程的切掉，剩下的干净窗口有多大？

In [ ]:
rows = []
for t in (-10, -5, -4, -3, -2.5, -2, -1.5):
    r = report(both[both.additive > t], f"可加预测 > {t}")
    r["test侧"] = int((both[both.additive > t].split == "test").sum())
    rows.append(r)
print(pd.DataFrame(rows)[["窗口", "n", "结构域", "test侧", "Pearson", "R2", "RMSE", "偏差", "残差SD"]].to_string(index=False))

print()
print("=== 干净窗口里，残差还跟位置间距有关吗 ===")
c = both[(both.additive > -2.5) & (both.ddg > -4.2)]
print(c.groupby(pd.cut(c.separation, [0, 4, 8, 16, 32, 10_000], labels=["1-4", "5-8", "9-16", "17-32", ">32"]),
                observed=True).residual.agg(["size", "mean", "std"]).round(3).to_string())

### 结果怎么看：切掉也救不回来

窗口收紧时发生了两件事，都对"做一个多突变模型"不利：

- **偏差变小但从不消失**（0.894 → 0.200），R² 最高只到 **0.175**。
- **相关系数反而下降**（0.807 → 0.578）。因为收紧窗口把动态范围一起切掉了——
  原来那个 0.81 的相关，大半是失稳尾巴这些**容易的样本**撑起来的。

而且干净窗口里，残差仍然**和位置间距无关**（1–4 是 0.348，>32 是 0.376）。
剩下的偏差还是系统性压缩，不是接触耦合。

### 面试怎么说
> "The apparent non-additivity is assay range compression: the residual scales with predicted
> severity at r = −0.72 and is flat in sequence separation, and doubles share the singles' −5.5 floor."

---

## 第 5 步 · 稳定端够不够做回溯验证

### 在干什么
数一数**实测确实更稳定**的突变有多少。

### 为什么这一步最要紧
压顶发生在**失稳端**。而"把蛋白改得更稳定"这件事在**另一端**。
第三阶段原本打算这样验证设计效果：
让模型提出一批稳定化突变，看留出的考试结构域里真正实测稳定的有几个落在前几名（precision@k）。
**这个做法成不成立，取决于稳定端有多少实测数据。**

### 执行

In [ ]:
print("=== 稳定端：可加预测 > 0 ===")
for lo in (0, 0.3, 0.5):
    f = both[both.additive > lo]
    x, y = f.additive.to_numpy(), f.ddg.to_numpy()
    print(f"  可加预测>{lo}: n={len(f):,}  结构域={f.WT_name.nunique()}  test侧={int((f.split == 'test').sum())}  "
          f"Pearson={stats.pearsonr(x, y)[0]:.3f}  偏差={f.residual.mean():+.3f}")

print()
print("=== 实测确实更稳定的双突变有多少 ===")
for t in (0, 0.5, 1.0):
    f = both[both.ddg > t]
    print(f"  实测 ddG > +{t}: {len(f):>7,} 行, {f.WT_name.nunique():>3} 个结构域, test 侧 {int((f.split == 'test').sum()):>5} 行")

print()
uniq = both[["WT_name", "wt0", "pos0", "mut0", "ddg0"]].drop_duplicates()
print(f"=== 单突变的候选空间：去重 {len(uniq):,} 个，其中实测 ddG>0 的 {int((uniq.ddg0 > 0).sum()):,} 个，>+0.5 的 {int((uniq.ddg0 > 0.5).sum()):,} 个 ===")

up = both[(both.ddg0 > 0) & (both.ddg1 > 0)]
x, y = up.additive.to_numpy(), up.ddg.to_numpy()
print()
print(f"=== 两个组分都稳定化的双突变：n={len(up):,} ===")
print(f"  Pearson={stats.pearsonr(x, y)[0]:.3f}  偏差={up.residual.mean():+.3f}  RMSE={np.sqrt(((y - x) ** 2).mean()):.3f}")
print(f"  实测真的比两者单独都更稳定的比例：{(up.ddg > up[['ddg0', 'ddg1']].max(axis=1)).mean():.1%}  <- 接近抛硬币")

### 结果怎么看

| | 数量 |
|---|---|
| 去重单突变里实测稳定化的（ΔΔG > 0） | **350 / 6,875**（5%） |
| 其中 > +0.5 kcal/mol | **79** |
| 实测 ΔΔG > +0.5 的双突变，**留出侧** | **173 行** |
| 实测 ΔΔG > +1.0 的双突变，**留出侧** | **20 行** |

而且稳定端的可加性本身就差：
- 两个组分都稳定化时，Pearson 只有 **0.525**，偏差翻成 **−0.21**（叠加起来**不如**两者之和，收益递减）
- **实测真的比两个组分单独都更稳定的，只有 53.7%——接近抛硬币**

### 关键结论
两条，都很硬：

1. **"把稳定化突变叠起来"这个设计循环的核心前提，在这批数据的双突变上得不到支持**（53.7%）。
2. **留出侧只有 20–173 行，散在 20–43 个结构域上，没有统计功效**支撑任何 precision@k 的说法。

### 面试怎么说
> "Only 5% of measured singles are stabilising and the held-out side has 173 rows above
> +0.5 kcal/mol, so precision-at-k on this data has no power."

---

## 这一章的结论

三条，按严重程度排：

### 1 · 湖里的两个能量就是考试集
"用已有打分训条件生成器"堵死。替代底座只能是 train split 蛋白的实测值。

### 2 · 多突变池看着大，实际被测量截断主导
可加性的"失败"（R² −0.42）**绝大部分是实验量程压顶**，不是上位效应：
残差和位置间距无关，和预测严重程度强相关（−0.72），双突变与单突变共用同一个 −5.5 地板。

把超量程的切掉，R² 最高只到 0.175，相关系数反而掉到 0.58。

**所以 15a 不能是"我们学会了上位效应"那一章。**
（补充：SPURS 已在**同一批** 22,081 行留出双突变上做过基准——学出来的模型 0.59，
自己的可加基线 0.52。头部空间只有 **0.07**，而且 MutateEverything 已经占了这个题。）

### 3 · 稳定端的数据撑不起回溯验证
见第 5 步。原定的 precision@k 基准**不成立**。

---

## 不换题目：否掉的是基准，不是生成式

上面第 3 条容易被读成"生成式做不了"。**那是错的**，写清楚免得以后自己看错。

否掉的是**一个特定的验证设计**。生成器本身（反向折叠、蛋白语言模型）这一章根本没碰到，
而且**功效在别处已经有了**：

- **14x 的靶场**：128 组实验、92 个考试蛋白、22 万个突变，已经证明过引导挑选的增益。
- **ddPCA 的双轴**：同一个文库同时密集测了蛋白量和结合，是现成的"稳定不掉、功能不丢"考卷。
- **生成式自己的标准指标**：序列恢复率、重折叠自洽、活性位点保留率，全部可算。

**这一章的真实产出是一条精确的边界，不是叫停**：
设计评估只在有实测的地方做，三个及以上突变的主张一律标成外推。

章节怎么切见 [`docs/related_work.md`](../../docs/related_work.md) 第 9 节。

## 新增的规矩（第三阶段起生效）

- **模型自己的输出不得作为下游模型的训练数据**，除非来源蛋白全部在 train split。
- **外部预训练模型算一个独立的泄漏面**：反向折叠模型训在 PDB 上（含考试蛋白结构），
  ESM-2 训在 UniRef 上（含考试蛋白序列）。现有切分保护的是打分器，不是生成器。
  每用一个就要给出"只用它、不用我们的模型"的消融，**而且消融就是过关标准**。
- **多突变的留出级别要写明**：是"没见过的蛋白"还是"见过的蛋白、没见过的组合"。

## 你现在要做什么

**什么都不用做。** 这一章是查验，没有要你操作的东西。
下一章 15a 开始才有真正要跑的东西（生成器），到时候会说清楚在哪跑、看什么。

## 记账

这一章没有训练模型，所有数字由本 notebook 自己从 `data/stability` 重算。
**MLflow 还没记**——等 15b 有了第一个真正的对照再一起记。